# KV grupitöö andmete analüüsimine ja mudeldamine

### Installime ja impordime vajalikud Pythoni teegid

In [37]:
!pip install pandas sqlalchemy
!pip install pandas
!pip install psycopg2-binary
!pip install statsmodels matplotlib seaborn

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ------------------- -------------------- 5.5/11.3 MB 28.6 MB/s eta 0:00:01
   ---------------------------------------  11.3/11.3 MB 29.1 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 26.3 MB/s  0:00:00

   ---------------------------------------- 0/5 [wrapt]
   -------- ------------------------------- 1/5 [patsy]
   -------- ------------------------------- 1/5 [patsy]
   -------- ------------------------------- 1/5 [patsy]
   -------- ------------------------------- 1/5 [patsy]
   ---------------- ----------------------- 2/5 [interface-meta]
   ------------------------ --------------- 3/5 [formulaic]
   ------------------------ --------------- 3/5 [formulaic]
   ------------------------ --------------- 3/5 [formulaic]
   ------------------------ --------------- 3/5 [formulaic]
   ------------------------ --------------- 3/5 [formulaic]
   ------------------------ --------------- 3/5 [fo

In [38]:
import pandas as pd
from sqlalchemy import create_engine, inspect
import statsmodels.api as sm
import seaborn as sn

### Tõmbame PostgreSQL'ist fact ja dim tabelid Pythonisse

In [19]:
# 1. PostgreSQL andmebaasi ühenduse andmed
DB_USER = "postgres"           # PostgreSQL vaikimisi kasutaja on tavaliselt postgres
DB_PASSWORD = "andmed"     
DB_HOST = "localhost"          # Kui andmebaas jookseb sinu arvutis
DB_PORT = "5432"               # PostgreSQL vaikimisi port
DB_NAME = "postgres"

In [20]:
# 2. Loo ühenduse mootor
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

In [21]:
# Sõnastik, kuhu kõik tabelid salvestatakse
tabelid = {}

In [22]:

# Määra skeemi nimi, kust andmeid soovid
SIHT_SCHEMA = "grupitoo_kv"

# Määra ära tabelid, mida skeemist soovid

vajalikud_tabelid = ['dim_county', 'fact_kv', 'dim_time', 'fact_palk', 'fact_intressid', 'fact_thi']

# Kood, mis tõmbab kõik tabelid korraga

try:
    with engine.connect() as connection:
        for tabeli_nimi in vajalikud_tabelid:
            query = f'SELECT * FROM "{SIHT_SCHEMA}"."{tabeli_nimi}"'
            
            # Tõmbab mällu ainult valitud tabeli
            tabelid[tabeli_nimi] = pd.read_sql_query(query, connection)
            print(f"✓ Vajalik tabel '{tabeli_nimi}' edukalt tõmmatud! Ridu: {len(tabelid[tabeli_nimi])}")
            
    print("\nValitud tabelid on edukalt mälus ja valmis kasutamiseks!")

except Exception as e:
    print(f"Tekkis viga: {e}. Kontrolli, kas tabelite nimed nimekirjas on kirjutatud õigesti!")

✓ Vajalik tabel 'dim_county' edukalt tõmmatud! Ridu: 15
✓ Vajalik tabel 'fact_kv' edukalt tõmmatud! Ridu: 1289
✓ Vajalik tabel 'dim_time' edukalt tõmmatud! Ridu: 86
✓ Vajalik tabel 'fact_palk' edukalt tõmmatud! Ridu: 1260
✓ Vajalik tabel 'fact_intressid' edukalt tõmmatud! Ridu: 86
✓ Vajalik tabel 'fact_thi' edukalt tõmmatud! Ridu: 84

Valitud tabelid on edukalt mälus ja valmis kasutamiseks!


### Eraldame kõik sissetõmmatud tabelid üksteisest

In [23]:
dim_time = tabelid["dim_time"]
dim_county = tabelid["dim_county"]
fact_kv = tabelid["fact_kv"]
fact_palk = tabelid["fact_palk"]
fact_thi = tabelid["fact_thi"]
fact_intressid = tabelid["fact_intressid"]

### Ühendame (joinime) fact ja dim tabelid

In [24]:
# 1. Alustame peamisest kinnisvara faktitabelist ja lisame sellele dimensioonid
df_koond = pd.merge(
    fact_kv,
    dim_county,
    left_on="fk_county_id",
    right_on="pk_county_id",
    how="left",
)
df_koond = pd.merge(
    df_koond, dim_time, left_on="fk_quarter_id", right_on="pk_quarter_id", how="left"
)

In [25]:
# 2. Lisame palga andmed (ühildame nii asukoha kui ka kvartali järgi)
df_koond = pd.merge(
    df_koond,
    fact_palk,
    on=["fk_county_id", "fk_quarter_id"],
    how="left",
    suffixes=("", "_palk"),
)

In [26]:
# 3. Lisame tarbijahinnaindeksi (THI) andmed (ühildame ainult kvartali järgi)
df_koond = pd.merge(
    df_koond, fact_thi, on="fk_quarter_id", how="left", suffixes=("", "_thi")
)

In [27]:
# 4. Lisame intresside andmed (ühildame ainult kvartali järgi)
df_koond = pd.merge(
    df_koond,
    fact_intressid,
    on="fk_quarter_id",
    how="left",
    suffixes=("", "_intress"),
)

In [28]:
# --- TULEMUSE KONTROLL ---
print(f"Koondtabeli kuju (ridu, veerge): {df_koond.shape}")
print("\nEsimesed 5 rida lõplikust analüüsitabelist:")
display(df_koond.head())

Koondtabeli kuju (ridu, veerge): (1289, 14)

Esimesed 5 rida lõplikust analüüsitabelist:


,fk_quarter_id,transaction_count,avg_price_m2,fk_county_id,pk_county_id,county_name,pk_quarter_id,year_number,quarter_number,quarter_string,half_year,average_salary,keskmine_indeks,intress
0,2005-01-01,2946,758.0,1,1,Harju maakond,2005-01-01,2005,1,2005 I,1,552.0,135.86,3.42
1,2005-01-01,8,163.0,2,2,Hiiu maakond,2005-01-01,2005,1,2005 I,1,400.0,135.86,3.42
2,2005-01-01,406,117.0,3,3,Ida-Viru maakond,2005-01-01,2005,1,2005 I,1,356.0,135.86,3.42
3,2005-01-01,52,132.0,5,5,Järva maakond,2005-01-01,2005,1,2005 I,1,390.0,135.86,3.42
4,2005-01-01,59,84.0,4,4,Jõgeva maakond,2005-01-01,2005,1,2005 I,1,394.0,135.86,3.42


##### Tabeli puhastamine (üleliigsete veergude eemaldamine)

In [31]:
# 1. Koosta nimekiri veergudest, mida sul analüüsiks ja graafikuteks vaja EI OLE
veerud_eemaldamiseks = [
    "fk_county_id",
    "pk_county_id",
    "pk_quarter_id",
    "quarter_string",
    "half_year"
]

In [32]:
# 2. Eemalda veerud (errors='ignore' hoiab ära veateate, kui mõnda veergu nimekirjas pole)
df_puhas = df_koond.drop(columns=veerud_eemaldamiseks, errors='ignore')

# 3. Kontrolli tulemust
print(f"Enne oli veere: {df_koond.shape[1]} | Nüüd on veere: {df_puhas.shape[1]}")
print("\nPuhastatud tabeli veerud:")
print(list(df_puhas.columns))

Enne oli veere: 14 | Nüüd on veere: 9

Puhastatud tabeli veerud:
['fk_quarter_id', 'transaction_count', 'avg_price_m2', 'county_name', 'year_number', 'quarter_number', 'average_salary', 'keskmine_indeks', 'intress']


In [34]:
# Muudame veerunimed ilusamaks
df_puhas = df_puhas.rename(columns={
    "transaction_count": "kinnisvara_tehingute_arv",
    "avg_price_m2": "keskmine_ruutmeetri_hind",
    "county_name": "maakond",
    "year_number": "aasta",
    "quarter_number": "kvartal",
    "average_salary": "keskmine_brutopalk",
    "keskmine_indeks": "tarbijahinnaindeks",
    "intress": "kodulaenu_intressimäär"
})

# Kuva esimesed read uuest puhtast tabelist
display(df_puhas.head())

,fk_quarter_id,kinnisvara_tehingute_arv,keskmine_ruutmeetri_hind,maakond,aasta,kvartal,keskmine_brutopalk,tarbijahinnaindeks,kodulaenu_intressimäär
0,2005-01-01,2946,758.0,Harju maakond,2005,1,552.0,135.86,3.42
1,2005-01-01,8,163.0,Hiiu maakond,2005,1,400.0,135.86,3.42
2,2005-01-01,406,117.0,Ida-Viru maakond,2005,1,356.0,135.86,3.42
3,2005-01-01,52,132.0,Järva maakond,2005,1,390.0,135.86,3.42
4,2005-01-01,59,84.0,Jõgeva maakond,2005,1,394.0,135.86,3.42


##### Tabeli puhastamine (tühjade väärtuste kontrollimine)

In [35]:
# Kuvab tühjade väärtuste summa iga veeru kohta
tyhjad_arvud = df_puhas.isnull().sum()
tyhjad_protsent = (df_puhas.isnull().sum() / len(df_puhas)) * 100

# Loome ülevaatliku tabeli
tyhjade_ylevaade = pd.DataFrame({'Tühje väärtusi': tyhjad_arvud, 'Protsent (%)': tyhjad_protsent})
display(tyhjade_ylevaade[tyhjade_ylevaade['Tühje väärtusi'] > 0])

,Tühje väärtusi,Protsent (%)
keskmine_ruutmeetri_hind,2,0.155159
keskmine_brutopalk,30,2.327386
tarbijahinnaindeks,30,2.327386


In [36]:
# .any(axis=1) otsib tühje väärtusi horisontaalselt üle kõigi veergude
df_tyhjad_read = df_puhas[df_puhas.isnull().any(axis=1)]

print(f"Kokku leiti {len(df_tyhjad_read)} rida, kus on tühje väärtusi.")
# Kuvab need read interaktiivse tabelina
display(df_tyhjad_read)

Kokku leiti 32 rida, kus on tühje väärtusi.


,fk_quarter_id,kinnisvara_tehingute_arv,keskmine_ruutmeetri_hind,maakond,aasta,kvartal,keskmine_brutopalk,tarbijahinnaindeks,kodulaenu_intressimäär
300,2010-01-01,3,NaN,Hiiu maakond,2010,1,610.0,171.35,3.59
540,2014-01-01,3,NaN,Hiiu maakond,2014,1,803.0,196.30,2.55
1259,2026-01-01,2701,2972.0,Harju maakond,2026,1,NaN,NaN,3.73
1260,2026-01-01,7,756.0,Hiiu maakond,2026,1,NaN,NaN,3.73
1261,2026-01-01,517,329.0,Ida-Viru maakond,2026,1,NaN,NaN,3.73
1262,2026-01-01,84,574.0,Järva maakond,2026,1,NaN,NaN,3.73
1263,2026-01-01,64,463.0,Jõgeva maakond,2026,1,NaN,NaN,3.73
1264,2026-01-01,70,1283.0,Lääne maakond,2026,1,NaN,NaN,3.73
1265,2026-01-01,189,722.0,Lääne-Viru maakond,2026,1,NaN,NaN,3.73
1266,2026-01-01,263,1835.0,Pärnu maakond,2026,1,NaN,NaN,3.73


In [39]:
# 1. Eemaldame ainult need read, kus korrelatsiooni jaoks kriitilised andmed on puudu, kasutades tabeli täpseid veerunimesi
df_analuys = df_puhas.dropna(subset=['keskmine_ruutmeetri_hind', 'keskmine_brutopalk', 'tarbijahinnaindeks'])

print(f"Algne ridade arv: {len(df_puhas)}")
print(f"Puhastatud ridade arv (mida kasutame analüüsis): {len(df_analuys)}")
print(f"Eemaldati kokku {len(df_puhas) - len(df_analuys)} rida, kus andmed olid puudulikud.")

# 2. Kontrolliks veendume, et uues tabelis on tühjade ridade arv null
print(f"Tühje ridu järele jäänud tabelis: {df_analuys.isnull().any(axis=1).sum()}")

print("-" * 50)

Algne ridade arv: 1289
Puhastatud ridade arv (mida kasutame analüüsis): 1257
Eemaldati kokku 32 rida, kus andmed olid puudulikud.
Tühje ridu järele jäänud tabelis: 0
--------------------------------------------------
